In [1]:
#Stream analysis
import pyxdf
import pandas as pd
import numpy as np
from datetime import datetime, timezone
import os
import time
import glob
from datetime import datetime


In [2]:
def inspect_marker_stream(streams):
    """Detailed inspection of DataSyncMarker stream to debug wall clock issues"""
    print("\n=== INSPECTING DATASYNCMARKER STREAM ===")
    
    marker_stream = None
    for stream in streams:
        if stream['info']['name'][0] == 'DataSyncMarker':
            marker_stream = stream
            break
    
    if marker_stream is None:
        print("DataSyncMarker stream not found!")
        return
    
    print(f"Stream found: {marker_stream['info']['name'][0]}")
    print(f"Channel count: {marker_stream['info']['channel_count'][0]}")
    
    # Examine time series data
    if 'time_series' in marker_stream and len(marker_stream['time_series']) > 0:
        time_series = np.array(marker_stream['time_series'])
        print(f"Time series shape: {time_series.shape}")
        
        # Check if it's a 2-channel stream
        if len(time_series.shape) > 1 and time_series.shape[1] == 2:
            print("\nExamining marker values (first channel):")
            markers = time_series[:, 0]
            unique_markers = np.unique(markers)
            print(f"Unique marker values: {unique_markers}")
            print(f"First few markers: {markers[:10]}")
            
            print("\nExamining wall clock times (second channel):")
            wall_times = time_series[:, 1]
            print(f"Wall clock range: {np.min(wall_times)} to {np.max(wall_times)}")
            print(f"First few wall clock values: {wall_times[:5]}")
            
            # Convert wall times to datetime for inspection
            datetime_times = [datetime.fromtimestamp(t).strftime('%Y-%m-%d %H:%M:%S.%f') 
                             for t in wall_times[:5]]
            print(f"Wall clock as datetime: {datetime_times}")
            
            # Check for any anomalies
            time_diffs = np.diff(wall_times)
            if np.any(time_diffs < 0):
                print("WARNING: Wall clock times are not monotonically increasing!")
                
            if np.any(wall_times < 946684800):  # Jan 1, 2000
                print("WARNING: Some wall clock times appear to be before year 2000!")
                
            if np.any(wall_times > 2524608000):  # Jan 1, 2050
                print("WARNING: Some wall clock times appear to be after year 2050!")
    
    print("=== END OF INSPECTION ===\n")

In [3]:
# Load the XDF file
file_path = "your_file_directory"  # Replace with your actual file path
streams, header = pyxdf.load_xdf(file_path)
# Call this function after loading your XDF file
#inspect_marker_stream(streams)

Stream 12: Calculated effective sampling rate 149.0366 Hz is different from specified rate 200.0000 Hz.


In [4]:
#Check streams
for idx, stream in enumerate(streams):
    print(f"Stream Index: {idx}")
    print(f"Source ID: {stream['info'].get('source_id', ['Unknown'])[0]}")
    print(f"Stream Name: {stream['info']['name'][0]}")
    print(f"Channel Count: {stream['info']['channel_count'][0]}")
    print(f"Sampling Rate: {stream['info']['nominal_srate'][0]} Hz\n")

Stream Index: 0
Source ID: None
Stream Name: Keyboard
Channel Count: 1
Sampling Rate: 0.000000000000000 Hz

Stream Index: 1
Source ID: MD-V5-0001023
Stream Name: PPG_RED
Channel Count: 1
Sampling Rate: 25.00000000000000 Hz

Stream Index: 2
Source ID: MD-V5-0001023
Stream Name: THERM
Channel Count: 1
Sampling Rate: 7.500000000000000 Hz

Stream Index: 3
Source ID: MD-V6-0000405
Stream Name: TEMP1
Channel Count: 1
Sampling Rate: 7.500000000000000 Hz

Stream Index: 4
Source ID: MD-V5-0001023
Stream Name: EDA
Channel Count: 1
Sampling Rate: 15.00000000000000 Hz

Stream Index: 5
Source ID: MD-V5-0001023
Stream Name: TEMP1
Channel Count: 1
Sampling Rate: 7.500000000000000 Hz

Stream Index: 6
Source ID: MD-V5-0001023
Stream Name: HR
Channel Count: 1
Sampling Rate: 0.000000000000000 Hz

Stream Index: 7
Source ID: MD-V6-0000405
Stream Name: PPG_IR
Channel Count: 1
Sampling Rate: 25.00000000000000 Hz

Stream Index: 8
Source ID: MD-V5-0001023
Stream Name: PPG_GRN
Channel Count: 1
Sampling Rate: 25

In [5]:
# Identify marker streams
marker_streams = []

for stream in streams:
    stream_type = stream['info']['type'][0].lower()

    if "marker" in stream_type or "keyboard" in stream_type:
        marker_streams.append(stream)  # Store all marker streams
        


In [6]:
# Function to extract marker timestamps and labels
def extract_markers(marker_stream):
    timestamps = marker_stream['time_stamps']
    labels = [m[0] for m in marker_stream['time_series']]  # Extract marker labels
    return timestamps, labels, marker_stream['info']['name'][0]  # Include stream name


# Extract markers with stream names
marker_data = [extract_markers(ms) for ms in marker_streams]

In [7]:
def export_all_streams_to_csv(file_path, output_dir):
    """
    Export all streams from the XDF file to CSV, including markers and their timestamps.
    Filter the Neon Companion_Neon Gaze stream to only include the Radius or Diameter channel.
    
    Parameters:
    file_path (str): Path to the XDF file
    output_dir (str): Directory to save CSV files
    """
    # Load the XDF file
    print(f"Loading XDF file: {file_path}")
    data, header = pyxdf.load_xdf(file_path)

    # Ensure the output directory exists
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Extract recording start time (created_at in Unix time)
    created_at = float(header["info"]["created_at"]) if "created_at" in header["info"] else None
    if created_at is None:
        print("Warning: 'created_at' field not found. Wall clock time conversion may be incorrect.")

    # IMPROVED STREAM IDENTIFICATION: Identify all marker/keyboard streams
    marker_streams = {}
    
    # IMPROVED KEYBOARD STREAM HANDLING
    keyboard_streams = []
    
    print("\n=== DETAILED STREAM INSPECTION FOR DEBUGGING ===")
    for idx, s in enumerate(data):
        stream_name = s['info']['name'][0]
        stream_type = s['info']['type'][0].lower() if 'type' in s['info'] else "unknown"
        
        # Print detailed information about each stream
        print(f"Stream {idx}: {stream_name} (Type: {stream_type})")
        print(f"  Channel count: {s['info']['channel_count'][0]}")
        print(f"  Time stamps count: {len(s['time_stamps'])}")
        print(f"  Time series shape: {np.array(s['time_series']).shape}")
        
        # Print a sample of time series data if available
        if len(s['time_series']) > 0:
            try:
                print(f"  First few values: {s['time_series'][:3]}")
            except:
                print(f"  Could not print sample values")
        
        # Check if this is a potential keyboard stream
        is_keyboard = False
        if (stream_name == 'Keyboard' or 
            'keyboard' in stream_name.lower() or 
            'key' in stream_type.lower()):
            
            is_keyboard = True
            print(f"  IDENTIFIED AS A KEYBOARD STREAM")
            
            # Check if it has data
            has_data = len(s['time_series']) > 0
            print(f"  Has data: {has_data}, time series length: {len(s['time_series'])}")
            
            # Add to our collection with metadata about data presence
            keyboard_streams.append({
                'stream': s,
                'name': stream_name,
                'has_data': has_data,
                'index': idx
            })
        
        # Check if this is any other marker type stream
        is_marker = False
        if 'marker' in stream_type.lower() or is_keyboard:
            is_marker = True
            print(f"  IDENTIFIED AS A MARKER STREAM")
        
        # Standard marker stream identification
        if stream_name == 'DataSyncMarker':
            marker_streams[stream_name] = s
        elif is_marker and not is_keyboard:  # Add non-keyboard marker streams
            marker_streams[stream_name] = s
    
    # Now select the best keyboard stream - prioritize non-empty streams
    primary_keyboard_stream = None
    if keyboard_streams:
        # First try to find a non-empty keyboard stream
        non_empty_streams = [ks for ks in keyboard_streams if ks['has_data']]
        if non_empty_streams:
            primary_keyboard_stream = non_empty_streams[0]['stream']
            print(f"Selected non-empty keyboard stream: {non_empty_streams[0]['name']} (index {non_empty_streams[0]['index']})")
        else:
            # Fall back to any keyboard stream if all are empty
            primary_keyboard_stream = keyboard_streams[0]['stream']
            print(f"All keyboard streams are empty, selected: {keyboard_streams[0]['name']} (index {keyboard_streams[0]['index']})")
        
        # Update the marker_streams dictionary to use only the primary keyboard stream
        marker_streams['Keyboard'] = primary_keyboard_stream
    else:
        print("No keyboard streams found!")
    
    print("=== END OF STREAM INSPECTION ===\n")
    
    # Debug check for Keyboard stream in marker_streams
    if 'Keyboard' in marker_streams:
        print("Found Keyboard stream in marker_streams dictionary")
        keyboard_str = marker_streams['Keyboard']
        print(f"Keyboard stream has {len(keyboard_str['time_stamps'])} timestamps")
        print(f"Keyboard time_series shape: {np.array(keyboard_str['time_series']).shape}")
        if len(keyboard_str['time_series']) > 0:
            print(f"First few keyboard values: {keyboard_str['time_series'][:5]}")
        else:
            print("WARNING: Keyboard stream time_series is empty!")
    else:
        print("WARNING: No Keyboard stream found in marker_streams!")

    # Loop through all streams and export
    for stream in data:
        stream_name = stream['info']['name'][0]

        # Skip exporting marker streams separately
        if stream_name in marker_streams:
            continue
        
        # Also skip if this is a keyboard stream that isn't our primary one
        is_keyboard_stream = (stream_name == 'Keyboard' or 
                             'keyboard' in stream_name.lower() or 
                             ('type' in stream['info'] and 'key' in stream['info']['type'][0].lower()))
        
        if is_keyboard_stream and primary_keyboard_stream is not None and stream != primary_keyboard_stream:
            print(f"Skipping non-primary keyboard stream: {stream_name}")
            continue

        stream_type = stream['info']['type'][0].lower()
        print(f"\nProcessing stream: {stream_name} (Type: {stream_type})")

        # Create DataFrame from the stream
        if stream_name == "Neon Companion_Neon Gaze":
            # Existing code for Neon_Gaze handling
            # ...
            # [Your existing Neon Gaze handling code - unchanged]
            print(f"Found Neon Companion_Neon Gaze stream - filtering to only Radius or Diameter channel")

            # First, let's print all available stream info to help with debugging
            print("Available stream info keys:", list(stream['info'].keys()))
            if 'desc' in stream['info']:
                print("Available desc keys:",
                      list(stream['info']['desc'][0].keys()) if stream['info']['desc'] else "No desc content")

            # Create a temporary full dataframe to examine channel characteristics
            temp_df = pd.DataFrame(
                stream['time_series'],
                columns=[f"{stream_name}_Channel_{i + 1}" for i in range(stream['time_series'].shape[1])],
                index=stream['time_stamps']
            )

            # Print stats for each channel to help identify the pupil diameter channel
            print("Channel statistics (first 5 rows):")
            print(temp_df.head())
            print("\nChannel ranges:")
            print(temp_df.describe().loc[['min', 'max']])

            # Check if we have channel information in the stream info
            radius_diameter_index = None
            channel_label = "Pupil_Diameter"

            if 'desc' in stream['info'] and stream['info']['desc'] and 'channels' in stream['info']['desc'][0]:
                print("Found channel metadata in stream info")
                channels_info = stream['info']['desc'][0]['channels'][0]['channel']

                print(f"Total channels in metadata: {len(channels_info)}")

                # Print all channel types to debug
                for i, channel_info in enumerate(channels_info):
                    if 'type' in channel_info:
                        channel_type = channel_info['type'][0]
                        channel_name = channel_info['label'][0] if 'label' in channel_info else f"Channel_{i + 1}"
                        print(f"  Channel {i}: {channel_name} - Type: {channel_type}")

                # Find the index of the Radius or Diameter channel
                for i, channel_info in enumerate(channels_info):
                    # Check if this channel is the Radius or Diameter channel
                    if 'type' in channel_info and channel_info['type'][0] == 'Radius or Diameter':
                        radius_diameter_index = i
                        channel_label = channel_info['label'][0] if 'label' in channel_info else f"Pupil_Diameter"
                        print(f"  Found Radius or Diameter channel at index {i} with label: {channel_label}")
                        break

            # If metadata-based search didn't work, use heuristics to find the pupil diameter channel
            if radius_diameter_index is None:
                print("Using heuristics to identify pupil diameter channel...")

                # Pupil diameter data typically has specific characteristics:
                # 1. Values usually between 2-8 mm for human pupils
                # 2. Less variation compared to position channels
                # 3. Always positive values

                # Calculate statistics for each channel
                channel_stats = temp_df.describe()
                potential_channels = []

                for i, col in enumerate(temp_df.columns):
                    col_data = temp_df[col]
                    col_mean = col_data.mean()
                    col_min = col_data.min()
                    col_max = col_data.max()
                    col_std = col_data.std()

                    # Check if data characteristics match pupil size data
                    # Most pupil diameters are between 2-8mm, but could be in pixels too
                    is_positive = col_min >= 0
                    has_reasonable_range = col_max < 1000  # Assuming it's not huge values

                    # Score each channel based on likelihood of being pupil data
                    score = 0
                    if is_positive: score += 1
                    if has_reasonable_range: score += 1
                    if 2 <= col_mean <= 20: score += 2  # Stronger weight if in typical range
                    if col_std / col_mean < 0.5: score += 1  # Pupil data varies but not wildly

                    potential_channels.append({
                        'index': i,
                        'column': col,
                        'score': score,
                        'mean': col_mean,
                        'min': col_min,
                        'max': col_max,
                        'std': col_std
                    })

                # Sort by score, highest first
                potential_channels.sort(key=lambda x: x['score'], reverse=True)

                # Print top candidates
                print("Top potential pupil diameter channels:")
                for i, channel in enumerate(potential_channels[:5]):
                    print(f"  Rank {i + 1}: {channel['column']} (Score: {channel['score']}) - " +
                          f"Mean: {channel['mean']:.2f}, Range: {channel['min']:.2f}-{channel['max']:.2f}")

                # Take the highest scoring channel
                if potential_channels:
                    best_channel = potential_channels[0]
                    radius_diameter_index = best_channel['index']
                    channel_label = "Pupil_Diameter"
                    print(f"Selected channel {radius_diameter_index + 1} as most likely pupil diameter")

            # Extract only the identified Radius or Diameter channel
            if radius_diameter_index is not None:
                print(f"Extracting only channel at index {radius_diameter_index}")
                pupil_diameter_data = stream['time_series'][:, radius_diameter_index]
                target_df = pd.DataFrame(
                    pupil_diameter_data.reshape(-1, 1),
                    columns=[channel_label],
                    index=stream['time_stamps']
                )
                print(f"Successfully extracted pupil diameter data to column: {channel_label}")
            else:
                print("WARNING: Could not identify the pupil diameter channel. Will maintain all channels.")
                # As a last resort, create DataFrame with all channels
                target_df = pd.DataFrame(
                    stream['time_series'],
                    columns=[f"{stream_name}_Channel_{i + 1}" for i in range(stream['time_series'].shape[1])],
                    index=stream['time_stamps']
                )
        else:
            # Handle other streams normally
            if len(stream['time_series'].shape) == 1:
                # Handle single channel data
                target_df = pd.DataFrame(
                    stream['time_series'].reshape(-1, 1),
                    columns=[stream_name],
                    index=stream['time_stamps']
                )
            else:
                # Handle multi-channel data
                target_df = pd.DataFrame(
                    stream['time_series'],
                    columns=[f"{stream_name}_Channel_{i + 1}" for i in range(stream['time_series'].shape[1])],
                    index=stream['time_stamps']
                )

        # MANUAL KEYBOARD COLUMN CREATION - Only add if no keyboard stream found
        if 'Keyboard' not in target_df.columns and 'Keyboard' not in marker_streams:
            print("Manually adding Keyboard column to DataFrame (no keyboard stream found)")
            target_df['Keyboard'] = None
            print("Successfully added empty Keyboard column")

        # Process each marker stream and merge into the data
        for marker_name, marker_stream in marker_streams.items():
            print(f"Adding marker data from stream: {marker_name}")
            
            # Skip if time_series is empty (handle the problem we're seeing)
            if len(marker_stream['time_series']) == 0:
                print(f"WARNING: {marker_name} stream has no time series data - skipping")
                
                # If this is the keyboard stream and we haven't added a Keyboard column yet
                if marker_name == 'Keyboard' and 'Keyboard' not in target_df.columns:
                    print("Creating empty Keyboard column as a placeholder")
                    target_df['Keyboard'] = None
                
                continue
            
            # Special handling for the Keyboard stream
            if marker_name == 'Keyboard':
                print(f"DEBUG - Processing primary Keyboard stream")
                print(f"  time_series shape: {np.array(marker_stream['time_series']).shape}")
                print(f"  time_stamps length: {len(marker_stream['time_stamps'])}")
                
                if len(marker_stream['time_series']) == 0:
                    print(f"  WARNING: Selected Keyboard stream has no data!")
                    target_df['Keyboard'] = None
                    continue
                
                # Print first few values for debugging
                print(f"  First few time series values: {marker_stream['time_series'][:5]}")
                
                # Extract keyboard data - handle different possible data structures
                marker_values = []
                for item in marker_stream['time_series']:
                    if isinstance(item, (list, np.ndarray)):
                        if len(item) > 0:
                            # Extract the value from the nested structure
                            marker_values.append(str(item[0]))  # Convert to string to handle different types
                        else:
                            marker_values.append(None)
                    else:
                        # Direct value (not nested)
                        marker_values.append(str(item) if item is not None else None)
                
                marker_array = np.array(marker_values)
                print(f"  Extracted {len(marker_array)} keyboard values")
                print(f"  Sample values: {marker_array[:5]}")
                
                # Create Series and add to DataFrame
                marker_series = pd.Series(
                    marker_array,
                    index=marker_stream['time_stamps'],
                    name=marker_name
                )
                
                # Debug - check what we've extracted
                print(f"  Created marker series with {len(marker_series)} values")
                print(f"  Non-null values: {marker_series.count()}")
                if marker_series.count() > 0:
                    print(f"  Sample values: {marker_series.dropna().head(5).tolist()}")
                
                # Merge with target DataFrame
                target_df = pd.concat([target_df, marker_series], axis=1)
                target_df = target_df.sort_index()
                
                # Forward fill with explicit debugging
                null_before = target_df[marker_name].isna().sum()
                target_df[marker_name] = target_df[marker_name].ffill()
                null_after = target_df[marker_name].isna().sum()
                print(f"  Forward filled: {null_before - null_after} null values filled")
                
            # Check if this is our DataSyncMarker stream with 2 channels
            elif marker_name == 'DataSyncMarker' and marker_stream['time_series'].shape[1] == 2:
                # Existing code for DataSyncMarker handling
                # ... [unchanged]
                print(f"  Detected 2-channel DataSyncMarker stream - extracting both marker and timestamp")

                # Extract marker values (first channel)
                marker_array = np.array(marker_stream['time_series'])[:, 0]
                
                # Extract timestamp values (second channel)
                timestamp_array = np.array(marker_stream['time_series'])[:, 1]
                
                # Filter out invalid timestamps (extremely small or future values)
                valid_indices = (timestamp_array > 946684800) & (timestamp_array < time.time() + 86400)
                
                if not np.any(valid_indices):
                    print("WARNING: No valid wall clock timestamps found. Using LSL timestamps instead.")
                    # Use LSL timestamps instead
                    marker_series = pd.Series(
                        marker_array,
                        index=marker_stream['time_stamps'],
                        name=marker_name
                    )
                    
                    # Approximate wall clock using recording start time
                    if 'created_at' in header['info']:
                        recording_start = float(header['info']['created_at'])
                        lsl_start = marker_stream['time_stamps'][0]
                        
                        # Convert LSL time to wall clock time
                        wall_clock_timestamps = recording_start + (marker_stream['time_stamps'] - lsl_start)
                        
                        timestamp_series = pd.Series(
                            wall_clock_timestamps,
                            index=marker_stream['time_stamps'],
                            name=f"{marker_name}_WallClock"
                        )
                    else:
                        print("WARNING: No created_at timestamp found. Using current time as reference.")
                        timestamp_series = pd.Series(
                            marker_stream['time_stamps'],
                            index=marker_stream['time_stamps'],
                            name=f"{marker_name}_WallClock"
                        )
                else:
                    print(f"  Found {np.sum(valid_indices)} valid wall clock timestamps out of {len(timestamp_array)}")
                    
                    # Filter the arrays to include only valid timestamps
                    filtered_marker_array = marker_array[valid_indices]
                    filtered_timestamp_array = timestamp_array[valid_indices]
                    filtered_time_stamps = marker_stream['time_stamps'][valid_indices]
                    
                    marker_series = pd.Series(
                        filtered_marker_array,
                        index=filtered_time_stamps,
                        name=marker_name
                    )
                    
                    timestamp_series = pd.Series(
                        filtered_timestamp_array,
                        index=filtered_time_stamps,
                        name=f"{marker_name}_WallClock"
                    )
                
                # Align timestamps and merge with target data
                target_df = pd.concat([target_df, marker_series, timestamp_series], axis=1)
                target_df = target_df.sort_index()
                
                # Forward fill missing markers and timestamps (using the non-deprecated methods)
                target_df[marker_name] = target_df[marker_name].ffill()
                target_df[f"{marker_name}_WallClock"] = target_df[f"{marker_name}_WallClock"].ffill()
                
                # Convert the wall clock timestamps to human-readable format in EST timezone
                # Convert Unix timestamp to datetime and then localize to EST
                target_df[f"{marker_name}_WallClock_Time"] = pd.to_datetime(
                    target_df[f"{marker_name}_WallClock"],
                    unit='s'
                ).dt.tz_localize('UTC').dt.tz_convert('US/Eastern')
                
                # Print diagnostic information
                if not target_df[f"{marker_name}_WallClock"].isna().all():
                    min_time = target_df[f"{marker_name}_WallClock"].min()
                    max_time = target_df[f"{marker_name}_WallClock"].max()
                    min_dt = pd.to_datetime(min_time, unit='s')
                    max_dt = pd.to_datetime(max_time, unit='s')
                    print(f"  Wall clock time range: {min_dt} to {max_dt}")
            else:
                # Regular single-channel marker stream handling
                marker_array = np.array(marker_stream['time_series']).flatten()
                marker_series = pd.Series(
                    marker_array,
                    index=marker_stream['time_stamps'],
                    name=marker_name
                )

                # Align timestamps and merge with target data
                target_df = pd.concat([target_df, marker_series], axis=1)
                target_df = target_df.sort_index()
                target_df[marker_name] = target_df[marker_name].ffill()  # Forward fill missing markers

        # Rest of the code remains unchanged
        # ...
        # [rest of the export_all_streams_to_csv function - unchanged]
        
        # Final check before exporting
        print("\nFinal check before exporting:")
        print(f"DataFrame shape: {target_df.shape}")
        print(f"Columns: {target_df.columns.tolist()}")
        
        if 'Keyboard' in target_df.columns:
            non_null_count = target_df['Keyboard'].count()
            print(f"Keyboard column has {non_null_count} non-null values out of {len(target_df)} rows")
            if non_null_count > 0:
                sample_values = target_df['Keyboard'].dropna().head(5).tolist()
                print(f"Sample Keyboard values: {sample_values}")
            else:
                print("WARNING: All Keyboard values are null!")
        else:
            print("WARNING: No Keyboard column in final DataFrame!")

        # Convert timestamp index to readable format
        target_df.reset_index(inplace=True)
        target_df.rename(columns={'index': 'temp_timestamp'}, inplace=True)

        # Calculate relative time without creating the Unix_Timestamp column
        start_time = target_df['temp_timestamp'].iloc[0]

        # Create time columns for calculations but they will be removed later
        temp_seconds_col = 'temp_seconds_since_start'
        target_df[temp_seconds_col] = target_df['temp_timestamp'] - start_time

        # Add video start/end event markers based on DataSyncMarker values
        if 'DataSyncMarker' in target_df.columns:
            # Create a new column for video event descriptions
            target_df['Video_Event'] = ''
            # Mark video start events (marker value = 1)
            target_df.loc[target_df['DataSyncMarker'] == 1, 'Video_Event'] = 'Video Start'
            # Mark video end events (marker value = 2)
            target_df.loc[target_df['DataSyncMarker'] == 2, 'Video_Event'] = 'Video End'

        # Ensure the DataSyncMarker_WallClock datetime is correctly propagated to all rows
        if 'DataSyncMarker_WallClock' in target_df.columns and 'DataSyncMarker_WallClock_Time' in target_df.columns:
            # Fill NaN values in DataSyncMarker_WallClock_Time column
            target_df['DataSyncMarker_WallClock_Time'] = target_df['DataSyncMarker_WallClock_Time'].ffill()

            # Store the datetime objects before converting to string
            # Explicitly convert timezone-aware datetimes to naive by removing timezone
            datetime_temp = target_df['DataSyncMarker_WallClock_Time'].dt.tz_localize(None)

            # Format date and time as strings that Excel will recognize
            # For dates, use MM/DD/YYYY format
            target_df['DataSyncMarker_Date'] = datetime_temp.dt.strftime('%m/%d/%Y')

            # For times, use HH:MM:SS format (24-hour)
            target_df['DataSyncMarker_Time'] = datetime_temp.dt.strftime('%H:%M:%S')

            # Now, compute the correct time for all rows based on DataSyncMarker=1 time
            # Find the first row where DataSyncMarker = 1 (video start)
            video_start_index = target_df.index[target_df['DataSyncMarker'] == 1].min() if 1 in target_df[
                'DataSyncMarker'].values else None

            if video_start_index is not None:
                print("Found video start marker. Computing accurate timestamps for all rows...")

                # Get the time at video start
                video_start_time = datetime_temp[video_start_index]

                # Get the seconds at video start
                video_start_seconds = target_df.loc[video_start_index, temp_seconds_col]

                # Create a new column for absolute time based on video start
                # Add a timedelta to the video start time for each row
                target_df['Absolute_Time'] = pd.to_datetime(
                    video_start_time + pd.to_timedelta(
                        target_df[temp_seconds_col] - video_start_seconds,
                        unit='s'
                    )
                )

                # Format the new absolute time as strings
                target_df['Absolute_Date'] = target_df['Absolute_Time'].dt.strftime('%m/%d/%Y')
                # Create a separate column for the time portion with milliseconds
                target_df['Absolute_Time_HMS'] = target_df['Absolute_Time'].dt.strftime('%H:%M:%S.%f').str[:-3]

                # Create simplified column names for final output
                target_df['Date'] = target_df['Absolute_Date']
                target_df['Time'] = target_df['Absolute_Time_HMS']
            else:
                print("No video start marker (DataSyncMarker=1) found. Cannot calculate absolute timestamps.")
                # Use DataSyncMarker date/time as fallback
                if 'DataSyncMarker_Date' in target_df.columns:
                    target_df['Date'] = target_df['DataSyncMarker_Date']
                if 'DataSyncMarker_Time' in target_df.columns:
                    target_df['Time'] = target_df['DataSyncMarker_Time']

        # Remove the redundant WallClock_Time column
        target_df = target_df.drop('DataSyncMarker_WallClock_Time', axis=1)

        # Remove the temporary timestamp column and any intermediate calculation columns
        columns_to_drop = [
            'temp_timestamp',
            temp_seconds_col,
            'Absolute_Time',
            'Absolute_Date',
            'Absolute_Time_HMS',
            'DataSyncMarker_Date',
            'DataSyncMarker_Time',
            'DataSyncMarker_WallClock',
            'Video_Event'
        ]

        # Only drop columns that actually exist
        columns_to_drop = [col for col in columns_to_drop if col in target_df.columns]

        target_df = target_df.drop(columns_to_drop, axis=1)

        # Reorder columns according to the specified order
        # First, ensure we have valid columns
        time_columns = []

        # Add Date and Time if they exist
        if 'Date' in target_df.columns:
            time_columns.append('Date')
        if 'Time' in target_df.columns:
            time_columns.append('Time')

        # Define marker columns (ensure they exist)
        marker_columns = []
        if 'Keyboard' in target_df.columns:
            marker_columns.append('Keyboard')
        if 'DataSyncMarker' in target_df.columns:
            marker_columns.append('DataSyncMarker')

        # Get all remaining columns (physiological data)
        data_columns = [col for col in target_df.columns if col not in time_columns and col not in marker_columns]

        # Create final column order
        column_order = time_columns + marker_columns + data_columns

        # Reorder the DataFrame
        target_df = target_df[column_order]

        # Export to CSV with Excel-friendly settings
        output_file = os.path.join(output_dir, f"exported_{stream_name}.csv")
        
        # Specify Excel-compatible formatting
        target_df.to_csv(output_file, index=False, float_format='%.6f')
        
        # Additionally, try to force Excel date recognition by creating a companion Excel file
        try:
            # Check if openpyxl is available before attempting to use it
            import importlib
            if importlib.util.find_spec("openpyxl") is not None:
                excel_output_file = os.path.join(output_dir, f"exported_{stream_name}.xlsx")
                
                # Create Excel writer with datetime format
                with pd.ExcelWriter(excel_output_file, engine='openpyxl', 
                                   datetime_format='yyyy-mm-dd') as writer:
                    # Write DataFrame to Excel with specific formatting
                    target_df.to_excel(writer, index=False, sheet_name='Data')
                    
                    # Get the workbook and sheet
                    workbook = writer.book
                    worksheet = writer.sheets['Data']
                    
                    # Format all date columns
                    if 'Date' in target_df.columns:
                        date_col_idx = target_df.columns.get_loc('Date') + 1  # Excel is 1-indexed
                        # Apply date formatting explicitly
                        for cell in worksheet[openpyxl.utils.get_column_letter(date_col_idx)][1:]:
                            cell.number_format = 'mm/dd/yyyy'
                    
                    # Format all time columns
                    if 'Time' in target_df.columns:
                        time_col_idx = target_df.columns.get_loc('Time') + 1
                        # Apply time formatting explicitly
                        for cell in worksheet[openpyxl.utils.get_column_letter(time_col_idx)][1:]:
                            cell.number_format = 'hh:mm:ss.000'  # Include milliseconds
                
                print(f"Excel file with formatted columns exported to {excel_output_file}")
        except ImportError:
            print("Warning: openpyxl package not available, Excel file not created.")
            print("Install with: pip install openpyxl")
        except Exception as e:
            print(f"Warning: Failed to create Excel file: {str(e)}")
        
        print(f"Data for stream '{stream_name}' exported to {output_file}")
    
    print("\nExport complete!")

In [8]:
def combine_exported_streams_with_downsampling(input_dir, output_file=None, target_freq=None, 
                                              window_size='100ms', agg_method='mean'):
    """
    Combine all exported CSV files from different streams into a single file with downsampling
    to handle different sampling rates.
    
    Parameters:
    input_dir (str): Directory containing the exported CSV files
    output_file (str, optional): Path for the combined output file. 
                               If None, will be named "combined_streams_downsampled.csv" in the input directory
    target_freq (float, optional): Target frequency in Hz for downsampling.
                                 If None, will use the lowest sampling rate from all streams.
    window_size (str): Size of the resampling window, e.g., '100ms', '1s'.
                       Only used if target_freq is None.
    agg_method (str): Aggregation method for downsampling ('mean', 'median', 'max', 'min', etc.)
    
    Returns:
    pandas.DataFrame: The combined downsampled dataframe
    """
    
    
    # Find all CSV files in the input directory
    csv_files = glob.glob(os.path.join(input_dir, "exported_*.csv"))
    
    if not csv_files:
        print(f"No exported stream files found in {input_dir}")
        return None
    
    print(f"Found {len(csv_files)} exported stream files to combine")
    
    # Initialize empty lists to hold dataframes and sampling rates
    all_dfs = []
    stream_names = []
    sampling_rates = []
    
    # Loop through all CSV files and load them
    for file_path in csv_files:
        # Extract stream name from filename
        stream_name = os.path.basename(file_path).replace("exported_", "").replace(".csv", "")
        print(f"Loading stream: {stream_name}")
        
        # Read the CSV
        stream_df = pd.read_csv(file_path)
        
        # Ensure Date and Time columns exist
        if 'Date' not in stream_df.columns or 'Time' not in stream_df.columns:
            print(f"Warning: {stream_name} is missing Date or Time columns. Skipping.")
            continue
        
        # Create a combined datetime column for sorting/merging
        try:
            stream_df['DateTime'] = pd.to_datetime(
                stream_df['Date'] + ' ' + stream_df['Time'], 
                format='%m/%d/%Y %H:%M:%S.%f'
            )
        except ValueError as e:
            print(f"  Warning: Error parsing datetime in {stream_name}: {e}")
            print(f"  Trying alternative datetime format...")
            try:
                # Try alternative format without milliseconds
                stream_df['DateTime'] = pd.to_datetime(
                    stream_df['Date'] + ' ' + stream_df['Time'], 
                    format='%m/%d/%Y %H:%M:%S'
                )
            except ValueError:
                print(f"  Error: Could not parse datetime in {stream_name}. Skipping this stream.")
                continue
        
        # Calculate sampling rate
        if len(stream_df) > 1:
            # Calculate time difference in seconds
            time_diff = (stream_df['DateTime'].iloc[-1] - stream_df['DateTime'].iloc[0]).total_seconds()
            if time_diff > 0:
                # Sampling rate = number of samples / time period
                rate = (len(stream_df) - 1) / time_diff
                sampling_rates.append(rate)
                print(f"  Estimated sampling rate: {rate:.2f} Hz")
            else:
                print(f"  Warning: Cannot determine sampling rate for {stream_name} (time span too short)")
                sampling_rates.append(None)
        else:
            print(f"  Warning: Not enough data points in {stream_name} to calculate sampling rate")
            sampling_rates.append(None)
            
        # Rename data columns to include stream name for clarity
        # But don't rename Date, Time, Keyboard, or DataSyncMarker columns
        reserved_columns = ['Date', 'Time', 'Keyboard', 'DataSyncMarker', 'DateTime']
        
        for col in stream_df.columns:
            if col not in reserved_columns:
                stream_df.rename(columns={col: f"{stream_name}_{col}"}, inplace=True)
        
        # Add to our lists
        all_dfs.append(stream_df)
        stream_names.append(stream_name)
        print(f"  Added {len(stream_df)} rows from {stream_name}")
    
    if not all_dfs:
        print("No valid data streams found.")
        return None
    
    # Determine target sampling rate if not provided
    if target_freq is None:
        # Filter out None values from sampling_rates
        valid_rates = [rate for rate in sampling_rates if rate is not None]
        
        if not valid_rates:
            print("Warning: Could not determine sampling rates. Using default window of '100ms'")
            freq = window_size
        else:
            # Calculate target frequency (use lowest rate to avoid oversampling)
            lowest_rate = min(valid_rates)
            
            # Convert to appropriate pandas frequency string
            # Round to nearest standard frequency
            if lowest_rate <= 0.1:  # Less than 0.1 Hz (slower than once per 10 seconds)
                freq = '10S'  # 10 seconds
            elif lowest_rate <= 1:  # Less than 1 Hz
                freq = '1S'   # 1 second
            elif lowest_rate <= 5:  # Less than 5 Hz
                freq = '200ms'  # 200 milliseconds
            elif lowest_rate <= 10:  # Less than 10 Hz
                freq = '100ms'  # 100 milliseconds
            elif lowest_rate <= 50:  # Less than 50 Hz
                freq = '20ms'   # 20 milliseconds
            elif lowest_rate <= 100:  # Less than 100 Hz
                freq = '10ms'   # 10 milliseconds
            else:  # Higher rates
                freq = '5ms'    # 5 milliseconds
                
            print(f"Automatically determined target frequency: approx. {lowest_rate:.2f} Hz ({freq})")
    else:
        # Convert Hz to milliseconds
        freq_ms = int(1000 / target_freq)
        freq = f'{freq_ms}ms'
        print(f"Using specified target frequency: {target_freq} Hz ({freq})")
    
    # Now downsample and combine all dataframes
    print(f"\nDownsampling all streams to common frequency: {freq}")
    resampled_dfs = []
    
    for i, df in enumerate(all_dfs):
        stream_name = stream_names[i]
        print(f"Downsampling {stream_name}...")
        
        # Set DateTime as index for resampling
        df = df.set_index('DateTime')
        
        # Identify marker and data columns
        marker_columns = [col for col in df.columns if col in ['Keyboard', 'DataSyncMarker']]
        data_columns = [col for col in df.columns if col not in marker_columns and col not in ['Date', 'Time']]
        
        # Create a new dataframe for the resampled data
        resampled_df = pd.DataFrame()
        
        # Resample data columns using the specified aggregation method
        if data_columns:
            # Convert non-numeric columns to numeric where possible
            for col in data_columns:
                if not np.issubdtype(df[col].dtype, np.number):
                    try:
                        df[col] = pd.to_numeric(df[col], errors='coerce')
                        print(f"  Converted {col} to numeric")
                    except:
                        print(f"  Could not convert {col} to numeric, will use mode for resampling")
            
            # Apply different aggregation methods based on column type
            numeric_cols = [col for col in data_columns if np.issubdtype(df[col].dtype, np.number)]
            other_cols = [col for col in data_columns if col not in numeric_cols]
            
            # Resample numeric columns with the specified aggregation method
            if numeric_cols:
                # Get data columns that can be aggregated numerically
                data_resampled = df[numeric_cols].resample(freq).agg(agg_method)
                resampled_df = pd.concat([resampled_df, data_resampled], axis=1)
            
            # For non-numeric columns, use 'first' or 'mode' aggregation
            if other_cols:
                # For categorical/string columns, use the most common value in each window
                other_resampled = df[other_cols].resample(freq).first()
                resampled_df = pd.concat([resampled_df, other_resampled], axis=1)
        
        # Handle marker columns specially - use 'max' to preserve markers
        for col in marker_columns:
            if col in df.columns:
                # For marker columns, we want to keep any non-NaN value
                # Convert to numeric if not already
                if not np.issubdtype(df[col].dtype, np.number):
                    try:
                        df[col] = pd.to_numeric(df[col], errors='coerce')
                    except:
                        # If conversion fails, just use the first value
                        marker_resampled = df[[col]].resample(freq).first()
                        resampled_df[col] = marker_resampled[col]
                        continue
                
                # Apply special logic for marker columns - use max to preserve markers
                marker_resampled = df[[col]].resample(freq).max()
                resampled_df[col] = marker_resampled[col]
        
        # Reset index to get DateTime back as a column
        resampled_df = resampled_df.reset_index()
        
        # Extract Date and Time from DateTime
        resampled_df['Date'] = resampled_df['DateTime'].dt.strftime('%m/%d/%Y')
        resampled_df['Time'] = resampled_df['DateTime'].dt.strftime('%H:%M:%S.%f').str[:-3]
        
        resampled_dfs.append(resampled_df)
        print(f"  Downsampled from {len(df)} to {len(resampled_df)} rows")
    
    # Now merge all resampled dataframes
    print("\nMerging downsampled streams...")
    
    # Start with the first dataframe
    combined_df = resampled_dfs[0]
    
    # Merge with all other dataframes
    for df in resampled_dfs[1:]:
        # Merge on DateTime
        combined_df = pd.merge(
            combined_df, 
            df, 
            on='DateTime', 
            how='outer',
            suffixes=('', '_duplicate')
        )
        
        # Handle duplicate columns (Date, Time, Keyboard, DataSyncMarker)
        # Keep the non-null values
        for col in combined_df.columns:
            if col.endswith('_duplicate'):
                original_col = col.replace('_duplicate', '')
                # Fill NaNs in the original column with values from the duplicate
                if original_col in combined_df.columns:
                    combined_df[original_col] = combined_df[original_col].fillna(combined_df[col])
                    combined_df = combined_df.drop(col, axis=1)
    
    # Sort by DateTime
    combined_df = combined_df.sort_values('DateTime')
    
    # Clean up the DataFrame
    # Move Date and Time columns to the front
    first_columns = ['Date', 'Time']
    if 'Keyboard' in combined_df.columns:
        first_columns.append('Keyboard')
    if 'DataSyncMarker' in combined_df.columns:
        first_columns.append('DataSyncMarker')
    
    # Get all other columns except DateTime
    other_columns = [col for col in combined_df.columns 
                    if col not in first_columns and col != 'DateTime']
    
    # Create final column order
    column_order = first_columns + other_columns
    
    # Reorder columns
    combined_df = combined_df[column_order]
    
    # Now drop the DateTime column if it exists
    if 'DateTime' in combined_df.columns:
        combined_df.drop('DateTime', axis=1, inplace=True)
    
    # Save to file if output_file is provided
    if output_file is None:
        output_file = os.path.join(input_dir, "combined_streams_downsampled.csv")
    
    combined_df.to_csv(output_file, index=False, float_format='%.6f')
    print(f"\nCombined data saved to: {output_file}")
    print(f"Final dataset contains {len(combined_df)} rows and {len(combined_df.columns)} columns")
    
    # Also save as Excel for better date/time formatting
    try:
        excel_output = output_file.replace('.csv', '.xlsx')
        combined_df.to_excel(excel_output, index=False)
        print(f"Excel version saved to: {excel_output}")
    except Exception as e:
        print(f"Warning: Could not create Excel file: {str(e)}")
    
    return combined_df

In [9]:
output_dir = "output_file_name"  # Directory to save CSV files

# Execute the export function
export_all_streams_to_csv(file_path, output_dir)

input_dir = output_dir  # Your exported data directory
combined_2_df = combine_exported_streams_with_downsampling(input_dir)

Loading XDF file: /Users/debbiehsu/Downloads/7_C.xdf


Stream 12: Calculated effective sampling rate 149.0366 Hz is different from specified rate 200.0000 Hz.



=== DETAILED STREAM INSPECTION FOR DEBUGGING ===
Stream 0: Keyboard (Type: markers)
  Channel count: 1
  Time stamps count: 242
  Time series shape: (242, 1)
  First few values: [['RETURN pressed'], ['RETURN released'], ['J pressed']]
  IDENTIFIED AS A KEYBOARD STREAM
  Has data: True, time series length: 242
  IDENTIFIED AS A MARKER STREAM
Stream 1: PPG_RED (Type: ppgred)
  Channel count: 1
  Time stamps count: 158332
  Time series shape: (158332, 1)
  First few values: [[129760.]
 [129746.]
 [129752.]]
Stream 2: THERM (Type: thermopile)
  Channel count: 1
  Time stamps count: 47630
  Time series shape: (47630, 1)
  First few values: [[32.179]
 [32.179]
 [32.191]]
Stream 3: TEMP1 (Type: temperature)
  Channel count: 1
  Time stamps count: 47644
  Time series shape: (47644, 1)
  First few values: [[36.077]
 [36.069]
 [36.101]]
Stream 4: EDA (Type: eda)
  Channel count: 1
  Time stamps count: 95258
  Time series shape: (95258, 1)
  First few values: [[0.030154]
 [0.030154]
 [0.030156]]

/var/folders/l2/43gk42tx7dj8f7ym4jwq7psm0000gn/T/ipykernel_76029/2974473453.py:174: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  data_resampled = df[numeric_cols].resample(freq).agg(agg_method)
/var/folders/l2/43gk42tx7dj8f7ym4jwq7psm0000gn/T/ipykernel_76029/2974473453.py:198: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  marker_resampled = df[[col]].resample(freq).max()
/var/folders/l2/43gk42tx7dj8f7ym4jwq7psm0000gn/T/ipykernel_76029/2974473453.py:198: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  marker_resampled = df[[col]].resample(freq).max()
/var/folders/l2/43gk42tx7dj8f7ym4jwq7psm0000gn/T/ipykernel_76029/2974473453.py:174: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  data_resampled = df[numeric_cols].resample(freq).agg(agg_method)
/var/folders/l2/43gk42tx7dj8f7ym4j

  Downsampled from 973545 to 6618 rows
Downsampling EDA...
  Downsampled from 95534 to 6353 rows
Downsampling PPG_IR...
  Downsampled from 158580 to 6354 rows
Downsampling PPG_RED...
  Downsampled from 158332 to 6353 rows

Merging downsampled streams...

Combined data saved to: 7_C_data/combined_streams_downsampled.csv
Final dataset contains 6618 rows and 12 columns


/var/folders/l2/43gk42tx7dj8f7ym4jwq7psm0000gn/T/ipykernel_76029/2974473453.py:198: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  marker_resampled = df[[col]].resample(freq).max()
/var/folders/l2/43gk42tx7dj8f7ym4jwq7psm0000gn/T/ipykernel_76029/2974473453.py:198: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  marker_resampled = df[[col]].resample(freq).max()
